# 05 Compare May 05 vs May 04 Default and Linear

Compares May 05 DRLB runs against the best May 04 default DRLB and the locked tuned LinearBidder baseline.

In [1]:
import sys
import json
import pickle
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
REPO_ROOT = cwd
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from example_notebooks.experiments.adapters.baseline_adapter import evaluate_baseline_model_inprocess
from example_notebooks.experiments.drlb.profiles import build_config as build_drlb_config
from example_notebooks.experiments.infra.split_utils import resolve_normalized_splits


/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Tuned linear params live under evaluate_baselines/best_params/<subfolder>/ (see baselines_finetune.BaseLineTrainer).
_linear_scr_fpa = Path(REPO_ROOT) / 'example_notebooks' / 'evaluate_baselines' / 'best_params' / 'fpa_baseline_n10_rndm_42' / 'linear_scr_FPA.pkl'
with _linear_scr_fpa.open('rb') as f:
    linear_tuned_params = pickle.load(f)
# Last expression must be at module level - Jupyter does not auto-display values inside with / if / etc.
linear_tuned_params


{'coef': 0.023335213830958296,
 'lower_clip': 9,
 'upper_clip': 1,
 'factor': 3.4224852046754637}

In [3]:
config_ref = build_drlb_config(
    run_name='may06_compare_may06_vs_may04_default_linear',
    profile='may05_default_best_fixed',
    split_set='full_train_val_holdout',
)
normalized_splits = resolve_normalized_splits(config_ref)

drlb_runs = {
    'may04_default_best': 'may04_default_state_optuna10',
    'may06_default_best_fixed': 'may06_default_best_fixed_baseline',
    'may06_linear_clip_lr_scheduler_search': 'may06_linear_clip_lr_scheduler_search',
    'may06_default_linear_clip_dqn_layer_norm_fixed': 'may06_default_linear_clip_dqn_layer_norm_fixed',
    'may06_default_linear_clip_dqn_layer_norm_scheduler_epsilon_search': 'may06_default_linear_clip_dqn_layer_norm_scheduler_epsilon_search',
}

summary_rows = []
for model, run_name in drlb_runs.items():
    summary_path = config_ref.family_dir / run_name / 'outputs' / 'run_summary.json'
    payload = json.loads(summary_path.read_text())
    summary_rows.append({
        'model': model,
        'run_name': run_name,
        'profile': payload.get('drlb_profile'),
        'best_params': payload['tuning']['best_params'],
        'val_clicks_sum': payload['tuning']['best_val_metrics']['clicks_sum'],
        'val_cpc_relative': payload['tuning']['best_val_metrics']['cpc_relative'],
        'val_rmse': payload['tuning']['best_val_metrics']['rmse'],
        'val_quickspend': payload['tuning']['best_val_metrics']['quickspend'],
        'holdout_clicks_sum': payload['final_holdout']['metrics']['clicks_sum'],
        'holdout_cpc_relative': payload['final_holdout']['metrics']['cpc_relative'],
        'holdout_rmse': payload['final_holdout']['metrics']['rmse'],
        'holdout_quickspend': payload['final_holdout']['metrics']['quickspend'],
        'holdout_end_balance_share': payload['final_holdout']['metrics'].get('average_end_balance_share'),
        'diagnostics_png': payload['refit']['combined_diagnostics_plot_path'],
    })

drlb_df = pd.DataFrame(summary_rows)
drlb_df


,model,run_name,profile,best_params,val_clicks_sum,val_cpc_relative,val_rmse,val_quickspend,holdout_clicks_sum,holdout_cpc_relative,holdout_rmse,holdout_quickspend,holdout_end_balance_share,diagnostics_png
0,may04_default_best,may04_default_state_optuna10,may04_default_linear_lambda_legacy,"{'dqn_lr': 0.0003, 'reward_net_lr': 0.01, 'bid...",2346.814328,623.640923,1.339748,0.027237,13013.699894,401.078269,1.730192,0.084047,0.474079,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
1,may06_default_best_fixed,may06_default_best_fixed_baseline,may06_default_best_fixed,{},1858.423906,119.222300,2.256330,0.132296,10587.023921,1192.135415,1.481849,0.049027,0.606279,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
2,may06_linear_clip_lr_scheduler_search,may06_linear_clip_lr_scheduler_search,may06_default_linear_clip_lr_scheduler_search,"{'dqn_gamma': 1.0, 'dqn_lr': 0.0003, 'reward_n...",2466.061439,1051.546352,1.249205,0.023346,13682.761130,755.713003,1.346361,0.022568,0.745619,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
3,may06_default_linear_clip_dqn_layer_norm_fixed,may06_default_linear_clip_dqn_layer_norm_fixed,may06_default_linear_clip_dqn_layer_norm_fixed,{},2147.862899,429.204946,1.531893,0.054475,7406.075293,3369.781990,1.236428,0.000778,0.922104,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
4,may06_default_linear_clip_dqn_layer_norm_sched...,may06_default_linear_clip_dqn_layer_norm_sched...,may06_default_linear_clip_dqn_layer_norm_sched...,"{'dqn_scheduler_name': 'exp_0_9999', 'dqn_epsi...",2341.254272,470.324250,1.511162,0.073930,2464.101459,116.151718,12.058564,0.279377,0.078874,/Users/amsafin/code/local_ml/rl/bat-autobiddin...


In [4]:
linear_val = evaluate_baseline_model_inprocess(
    model_name='linear',
    label='linear_val',
    params_dict=linear_tuned_params,
    split=normalized_splits['val'],
    auction_mode='FPA',
)
linear_holdout = evaluate_baseline_model_inprocess(
    model_name='linear',
    label='linear_holdout',
    params_dict=linear_tuned_params,
    split=normalized_splits['test_holdout'],
    auction_mode='FPA',
)

linear_row = {
    'model': 'linear_tuned',
    'run_name': 'fpa_baseline_n10_rndm_42',
    'profile': 'linear_scr_FPA',
    'best_params': linear_tuned_params,
    'val_clicks_sum': linear_val['metrics']['clicks_sum'],
    'val_cpc_relative': linear_val['metrics']['cpc_relative'],
    'val_rmse': linear_val['metrics']['rmse'],
    'val_quickspend': linear_val['metrics']['quickspend'],
    'holdout_clicks_sum': linear_holdout['metrics']['clicks_sum'],
    'holdout_cpc_relative': linear_holdout['metrics']['cpc_relative'],
    'holdout_rmse': linear_holdout['metrics']['rmse'],
    'holdout_quickspend': linear_holdout['metrics']['quickspend'],
    'holdout_end_balance_share': linear_holdout['metrics'].get('average_end_balance_share'),
    'diagnostics_png': None,
}

comparison_df = pd.concat([pd.DataFrame([linear_row]), drlb_df], ignore_index=True)
comparison_df


/var/folders/ht/mcd64cts6p959c6g8xqp_jh00000gn/T/ipykernel_81394/728431707.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  comparison_df = pd.concat([pd.DataFrame([linear_row]), drlb_df], ignore_index=True)


,model,run_name,profile,best_params,val_clicks_sum,val_cpc_relative,val_rmse,val_quickspend,holdout_clicks_sum,holdout_cpc_relative,holdout_rmse,holdout_quickspend,holdout_end_balance_share,diagnostics_png
0,linear_tuned,fpa_baseline_n10_rndm_42,linear_scr_FPA,"{'coef': 0.023335213830958296, 'lower_clip': 9...",3729.287568,396.173121,1.407279,0.003891,17792.733546,421.338042,1.503115,0.004669,NaN,None
1,may04_default_best,may04_default_state_optuna10,may04_default_linear_lambda_legacy,"{'dqn_lr': 0.0003, 'reward_net_lr': 0.01, 'bid...",2346.814328,623.640923,1.339748,0.027237,13013.699894,401.078269,1.730192,0.084047,0.474079,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
2,may06_default_best_fixed,may06_default_best_fixed_baseline,may06_default_best_fixed,{},1858.423906,119.222300,2.256330,0.132296,10587.023921,1192.135415,1.481849,0.049027,0.606279,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
3,may06_linear_clip_lr_scheduler_search,may06_linear_clip_lr_scheduler_search,may06_default_linear_clip_lr_scheduler_search,"{'dqn_gamma': 1.0, 'dqn_lr': 0.0003, 'reward_n...",2466.061439,1051.546352,1.249205,0.023346,13682.761130,755.713003,1.346361,0.022568,0.745619,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
4,may06_default_linear_clip_dqn_layer_norm_fixed,may06_default_linear_clip_dqn_layer_norm_fixed,may06_default_linear_clip_dqn_layer_norm_fixed,{},2147.862899,429.204946,1.531893,0.054475,7406.075293,3369.781990,1.236428,0.000778,0.922104,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
5,may06_default_linear_clip_dqn_layer_norm_sched...,may06_default_linear_clip_dqn_layer_norm_sched...,may06_default_linear_clip_dqn_layer_norm_sched...,"{'dqn_scheduler_name': 'exp_0_9999', 'dqn_epsi...",2341.254272,470.324250,1.511162,0.073930,2464.101459,116.151718,12.058564,0.279377,0.078874,/Users/amsafin/code/local_ml/rl/bat-autobiddin...


In [5]:
linear_clicks = float(comparison_df.loc[comparison_df['model'] == 'linear_tuned', 'holdout_clicks_sum'].iloc[0])
may04_clicks = float(comparison_df.loc[comparison_df['model'] == 'may04_default_best', 'holdout_clicks_sum'].iloc[0])

delta_df = comparison_df.copy()
for split in ['val', 'holdout']:
    for metric in ['clicks_sum', 'cpc_relative', 'rmse', 'quickspend']:
        col = f'{split}_{metric}'
        linear_value = float(comparison_df.loc[comparison_df['model'] == 'linear_tuned', col].iloc[0])
        may04_value = float(comparison_df.loc[comparison_df['model'] == 'may04_default_best', col].iloc[0])
        delta_df[f'{col}_delta_vs_linear'] = delta_df[col] - linear_value
        delta_df[f'{col}_delta_vs_may04_default'] = delta_df[col] - may04_value

delta_df.sort_values('holdout_clicks_sum', ascending=False)


,model,run_name,profile,best_params,val_clicks_sum,val_cpc_relative,val_rmse,val_quickspend,holdout_clicks_sum,holdout_cpc_relative,...,val_quickspend_delta_vs_linear,val_quickspend_delta_vs_may04_default,holdout_clicks_sum_delta_vs_linear,holdout_clicks_sum_delta_vs_may04_default,holdout_cpc_relative_delta_vs_linear,holdout_cpc_relative_delta_vs_may04_default,holdout_rmse_delta_vs_linear,holdout_rmse_delta_vs_may04_default,holdout_quickspend_delta_vs_linear,holdout_quickspend_delta_vs_may04_default
0,linear_tuned,fpa_baseline_n10_rndm_42,linear_scr_FPA,"{'coef': 0.023335213830958296, 'lower_clip': 9...",3729.287568,396.173121,1.407279,0.003891,17792.733546,421.338042,...,0.000000,-0.023346,0.000000,4779.033652,0.000000,20.259773,0.000000,-0.227077,0.000000,-0.079377
3,may06_linear_clip_lr_scheduler_search,may06_linear_clip_lr_scheduler_search,may06_default_linear_clip_lr_scheduler_search,"{'dqn_gamma': 1.0, 'dqn_lr': 0.0003, 'reward_n...",2466.061439,1051.546352,1.249205,0.023346,13682.761130,755.713003,...,0.019455,-0.003891,-4109.972416,669.061236,334.374961,354.634734,-0.156755,-0.383831,0.017899,-0.061479
1,may04_default_best,may04_default_state_optuna10,may04_default_linear_lambda_legacy,"{'dqn_lr': 0.0003, 'reward_net_lr': 0.01, 'bid...",2346.814328,623.640923,1.339748,0.027237,13013.699894,401.078269,...,0.023346,0.000000,-4779.033652,0.000000,-20.259773,0.000000,0.227077,0.000000,0.079377,0.000000
2,may06_default_best_fixed,may06_default_best_fixed_baseline,may06_default_best_fixed,{},1858.423906,119.222300,2.256330,0.132296,10587.023921,1192.135415,...,0.128405,0.105058,-7205.709625,-2426.675973,770.797372,791.057145,-0.021266,-0.248343,0.044358,-0.035019
4,may06_default_linear_clip_dqn_layer_norm_fixed,may06_default_linear_clip_dqn_layer_norm_fixed,may06_default_linear_clip_dqn_layer_norm_fixed,{},2147.862899,429.204946,1.531893,0.054475,7406.075293,3369.781990,...,0.050584,0.027237,-10386.658253,-5607.624601,2948.443948,2968.703721,-0.266687,-0.493764,-0.003891,-0.083268
5,may06_default_linear_clip_dqn_layer_norm_sched...,may06_default_linear_clip_dqn_layer_norm_sched...,may06_default_linear_clip_dqn_layer_norm_sched...,"{'dqn_scheduler_name': 'exp_0_9999', 'dqn_epsi...",2341.254272,470.324250,1.511162,0.073930,2464.101459,116.151718,...,0.070039,0.046693,-15328.632087,-10549.598435,-305.186325,-284.926552,10.555449,10.328372,0.274708,0.195331
